# Conventionality Evaluator

**The Conventionality Evaluator** measures how explicit, literal, and straightforward a text's meaning is for students in grades 3–11. It assesses whether meaning is on the surface or requires abstract reasoning, interpretation of figurative language, or familiarity with unconventional expressions. When you run a passage through the evaluator, it returns a structured output that includes:

* **complexity_score**: The conventionality complexity level (Slightly to Exceedingly Complex).
* **conventionality_features**: Specific language features driving the complexity (e.g., idioms, metaphors, implied meaning) with direct quotes from the text.
* **grade_context**: How the conventionality demands compare to general expectations for the target grade.
* **instructional_insights**: Actionable pedagogical suggestions for scaffolding the unconventional language features in the classroom.
* **reasoning**: A synthesis of why the text fits the chosen complexity level.

This gives you a clear signal about the figurative and abstract language demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

### Install & Load necessary packages

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic textstat

In [ ]:
import getpass
import os
import sys
from pathlib import Path
from typing import List, Literal

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from textstat import textstat as ts

_scripts_dir = None
for _root in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    _candidate = _root / "sdks" / "python" / "scripts"
    if (_candidate / "capture.py").is_file():
        _scripts_dir = _candidate
        break
if _scripts_dir is None:
    raise RuntimeError(
        "Could not find sdks/python/scripts/capture.py. Run this notebook from a "
        "checkout of the evaluators repo (cwd may be repo root, evals/, sdks/python/, "
        "or any subdirectory under the repo)."
    )
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))
from capture import capture_llm, capture_case, reset_captures, build_contract_toml

### Set up the evaluator's model and prompts

In [ ]:
from prompts import conventionality_prompts as prompts

# Set your api key in your environment, .env file, or enter when prompted.
# os.environ['GOOGLE_API_KEY'] = 'YOUR API KEY'
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

MODEL_NAME = "gemini-3-flash-preview"
TEMPERATURE = 0
model = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=TEMPERATURE)

### Set up the output structure

In [ ]:
class ConventionalityOutput(BaseModel):
    conventionality_features: List[str] = Field(
        description="List of the specific language features driving the complexity (e.g., idioms, metaphors, implied meaning) with direct quotes from the text."
    )
    grade_context: str = Field(
        description="How the conventionality demands compare to general expectations for the provided target grade."
    )
    instructional_insights: str = Field(
        description="Actionable pedagogical suggestions for scaffolding the unconventional language features in the classroom."
    )
    complexity_score: Literal[
        "slightly_complex",
        "moderately_complex",
        "very_complex",
        "exceedingly_complex"
    ] = Field(description="The conventionality complexity level of the text")
    reasoning: str = Field(
        description="A synthesis of why the text fits the chosen rubric level."
    )


prompt_vars = {
    "inputVars": ["text", "grade", "fk_score"],
    "outputParser": JsonOutputParser(pydantic_object=ConventionalityOutput),
}

### Define text complexity evaluation function

In [ ]:
def calculate_fk_score(text) -> float:
    """
    Calculate the Flesch-Kincaid Grade Level
    """
    fk_score = round(ts.flesch_kincaid_grade(text), 2)

    return fk_score


def predict_text_complexity_level(text, grade):
    dataset = {
        "text": text,
        "grade": grade,
        "fk_score": calculate_fk_score(text),
    }

    messages = [
        SystemMessage(content=prompts.conventionality_system_prompt),
        HumanMessagePromptTemplate.from_template(prompts.conventionality_user_prompt),
    ]

    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_vars["inputVars"],
        partial_variables={
            "format_instructions": prompt_vars["outputParser"].get_format_instructions()
        },
    )

    chain = prompt | capture_llm("main", model) | JsonOutputParser()
    return chain.invoke(dataset)

# Test out examples

In [ ]:
# Add your text & the grade level you want to evaluate for conventionality complexity

sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse for half an hour. Think of something to write about, and write the word on your slate. Then try to tell what it is, what it is like, what it is good for, and what is done with it. That is the way to write a composition." Henry took his slate and went out. Just behind the schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden, Henry saw a turnip. "Well, I know what that is," he said to himself; and he wrote the word turnip on his slate. Then he tried to tell what it was like, what it was good for, and what was done with it. Before the half hour was ended he had written a very neat composition on his slate. He then went into the house, and waited while the teacher read it. The teacher was surprised and pleased. He said, "Henry Longfellow, you have done very well. Today you may stand up before the school and read what you have written about the turnip."
"""

result = predict_text_complexity_level(sample_text, 4)
display(result)

In [ ]:
reset_captures()

sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse for half an hour. Think of something to write about, and write the word on your slate. Then try to tell what it is, what it is like, what it is good for, and what is done with it. That is the way to write a composition." Henry took his slate and went out. Just behind the schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden, Henry saw a turnip. "Well, I know what that is," he said to himself; and he wrote the word turnip on his slate. Then he tried to tell what it was like, what it was good for, and what was done with it. Before the half hour was ended he had written a very neat composition on his slate. He then went into the house, and waited while the teacher read it. The teacher was surprised and pleased. He said, "Henry Longfellow, you have done very well. Today you may stand up before the school and read what you have written about the turnip."
"""
input = {"text": sample_text, "grade": 4}
result = predict_text_complexity_level(**input)

capture = capture_case(
    name="turnip",
    description="Grade 4 classroom narrative (Henry and the turnip)",
    input=input,
    llm_call_captures=["main"],
    expected_result=result,
)

print(build_contract_toml(capture))

You can copy or edit the above cell to test out different texts and grade levels.